# Arbitrary States — Qiskit Implementation

**Primitive:** Arbitrary single-qubit state preparation via RY-RZ rotations

**SETUP (assumed defaults — edit the angles below if your report needs different values)**
- N QUBITS: 2 (prepared independently, no entanglement)
- QUBIT 0: RZ(phi0) . RY(theta0), theta0 = pi/3, phi0 = pi/4
- QUBIT 1: RZ(phi1) . RY(theta1), theta1 = 2*pi/3, phi1 = pi/2
- QUBIT ORDERING: qubit 0 = rightmost bit in printed bitstring (Qiskit's native convention)
- SHOTS: 10000, SEED: 42
- GLOBAL PHASE: normalized so first nonzero amplitude (ascending bitstring order) is real and positive

**Theory recap:** Any single-qubit pure state can be written (up to global phase) as
|psi> = RZ(phi) RY(theta) |0>, where RY(theta) sets the polar angle on the Bloch sphere
(controls the |0>/|1> amplitude split) and RZ(phi) sets the relative phase between them.
This is the standard gate-level (non-opaque) way to prepare an arbitrary qubit state,
as opposed to calling a library `initialize()` routine.

RY(theta) = [[cos(theta/2), -sin(theta/2)], [sin(theta/2), cos(theta/2)]]

RZ(phi)   = [[exp(-i phi/2), 0], [0, exp(i phi/2)]]

**IMPORTANT: Run every cell in order, top to bottom** (Kernel -> Restart & Run All).

In [ ]:
# Run this cell first if you get ModuleNotFoundError or MissingOptionalLibraryError.
# pylatexenc is required for circuit.draw(output="mpl") gate-label rendering -- not pulled in
# automatically by the base qiskit install, so it must be listed explicitly.
!pip install -q qiskit qiskit-aer pylatexenc
# Restart the kernel after this finishes (Kernel -> Restart Kernel), then Run All.


In [ ]:
# --- Imports ---
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# --- SETUP constants (all cells below use these -- defined once, here) ---
N_QUBITS = 2
SHOTS = 10000
SEED = 42

# Rotation angles: (theta, phi) per qubit, applied as RZ(phi) . RY(theta)
ANGLES = {
    0: (np.pi / 3, np.pi / 4),        # qubit 0: theta0=pi/3,   phi0=pi/4
    1: (2 * np.pi / 3, np.pi / 2),    # qubit 1: theta1=2pi/3,  phi1=pi/2
}
print("Rotation angles (theta, phi) per qubit:", ANGLES)

## 1. Build the circuit explicitly, gate by gate

In [ ]:
qc = QuantumCircuit(N_QUBITS, N_QUBITS, name="arbitrary_states")

for q in range(N_QUBITS):
    theta, phi = ANGLES[q]
    qc.ry(theta, q)  # Arbitrary-state prep: RY sets the |0>/|1> amplitude split (polar angle)
    qc.rz(phi, q)    # Arbitrary-state prep: RZ sets the relative phase between |0> and |1>

qc.barrier()
qc.draw(output="mpl")

## 2. Draw the circuit and save to file

In [ ]:
fig = qc.draw(output="mpl")
fig.savefig("arbitrary_states_qiskit_circuit.png", bbox_inches="tight")
fig

## 3. Explicit matrix representation of the key operator(s)

In [ ]:
def ry_matrix(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s],
                      [s,  c]], dtype=complex)

def rz_matrix(phi):
    return np.array([[np.exp(-1j * phi / 2), 0],
                      [0, np.exp(1j * phi / 2)]], dtype=complex)

for q in range(N_QUBITS):
    theta, phi = ANGLES[q]
    print(f"--- Qubit {q}: theta={theta:.4f}, phi={phi:.4f} ---")
    print("RY(theta) matrix:")
    print(np.round(ry_matrix(theta), 4))
    print("RZ(phi) matrix:")
    print(np.round(rz_matrix(phi), 4))
    combined = rz_matrix(phi) @ ry_matrix(theta)
    print("Combined single-qubit operator RZ(phi).RY(theta):")
    print(np.round(combined, 4))
    print()

In [ ]:
prep_circuit = QuantumCircuit(N_QUBITS, name="prep_only")
for q in range(N_QUBITS):
    theta, phi = ANGLES[q]
    prep_circuit.ry(theta, q)
    prep_circuit.rz(phi, q)

full_op = Operator(prep_circuit)
print(f"Full {2**N_QUBITS}x{2**N_QUBITS} operator (basis {{|00>, |01>, |10>, |11>}}):")
print(np.round(full_op.data, 4))

## 4. Exact statevector (before measurement), global-phase normalized

In [ ]:
sv = Statevector.from_instruction(qc)
amps = sv.data.copy()

def normalize_global_phase(statevector_data, n_qubits):
    dim = 2 ** n_qubits
    bitstrings = [format(i, f"0{n_qubits}b") for i in range(dim)]
    order = np.argsort(bitstrings)
    for idx in order:
        amp = statevector_data[idx]
        if abs(amp) > 1e-9:
            phase = np.angle(amp)
            corrected = statevector_data * np.exp(-1j * phase)
            return corrected, bitstrings
    return statevector_data, bitstrings

amps_norm, bitstrings = normalize_global_phase(amps, N_QUBITS)

exact_probs = {}
for i, bs in enumerate(bitstrings):
    exact_probs[bs] = round(abs(amps_norm[i]) ** 2, 4)

exact_probs = dict(sorted(exact_probs.items()))
print("Exact statevector amplitudes (phase-normalized):")
print(np.round(amps_norm, 4))
print()
print("Exact statevector probabilities (qubit0 = rightmost):")
print(exact_probs)

**Ordering note:** Qiskit's `Statevector` already indexes amplitudes with qubit 0 as the
least-significant (rightmost) bit of the binary index -- matches the SETUP convention
directly, so no reversal was needed here.

## 5. Measurement: add measurements, run AerSimulator, plot histogram

In [ ]:
qc_measured = qc.copy()
qc_measured.measure(range(N_QUBITS), range(N_QUBITS))

sim = AerSimulator(seed_simulator=SEED)
job = sim.run(qc_measured, shots=SHOTS, seed_simulator=SEED)
result = job.result()
counts = result.get_counts()

fig2 = plot_histogram(counts)
fig2.savefig("arbitrary_states_qiskit_histogram.png", bbox_inches="tight")
fig2

In [ ]:
sampled_probs = {bs: round(c / SHOTS, 4) for bs, c in counts.items()}
sampled_probs = dict(sorted(sampled_probs.items()))
print(f"Sampled probabilities ({SHOTS} shots, seed={SEED}):")
print(sampled_probs)

## 6. Circuit resource counts

In [ ]:
print(f"Depth: {qc_measured.depth()}")
print(f"Gate/operation count: {qc_measured.size()}")

## 7. Convention notes

In [ ]:
print("Bit ordering: Qiskit's Statevector and get_counts() both natively use qubit 0 as the")
print("rightmost (least-significant) bit of the printed bitstring, matching this notebook's")
print("SETUP convention exactly -> NO reversal applied.")
print()
print("Global phase: normalized by finding the first nonzero amplitude in ascending-bitstring")
print("order and rotating the whole statevector so that amplitude becomes real and positive.")
print("Unlike Basic States or Superposition, this primitive produces genuinely complex")
print("amplitudes (due to the RZ phase rotation), so this normalization step is NOT a no-op")
print("here -- it materially changes the printed amplitude values versus the raw statevector.")

## Final result (for diffing against the CUDA-Q version)

In [ ]:
QISKIT_RESULT = exact_probs
print("QISKIT_RESULT =", QISKIT_RESULT)